# 104 - Pipeline — **Soluciones + Autotests**

In [ ]:

import pandas as pd
from pathlib import Path

HAS_PREF=False
try:
    from prefect import flow, task
    HAS_PREF=True
except Exception:
    pass

if HAS_PREF:
    @task
    def extract(): return pd.DataFrame({'a':[1,2,None],'b':[3,4,5]})
    @task
    def transform(df): return df.dropna()
    @task
    def load(df): Path('artifacts').mkdir(exist_ok=True); path='artifacts/clean.csv'; df.to_csv(path, index=False); return path
    @flow
    def etl(): return load(transform(extract()))
    result_path = etl()
else:
    def etl():
        df = pd.DataFrame({'a':[1,2,None],'b':[3,4,5]})
        df = df.dropna()
        Path('artifacts').mkdir(exist_ok=True)
        path='artifacts/clean.csv'
        df.to_csv(path, index=False)
        return path
    result_path = etl()


In [ ]:

# Helper de autotests
def check(cond, ok_msg, fail_msg):
    if cond:
        print("✔", ok_msg)
    else:
        raise AssertionError("✘ " + fail_msg)


In [ ]:

# Tests
from pathlib import Path
check(Path(result_path).exists(), "Pipeline generó artifacts/clean.csv.", "No se encontró el CSV limpio.")
import pandas as pd
dfc = pd.read_csv(result_path)
check('a' in dfc.columns and 'b' in dfc.columns, "CSV limpio con columnas esperadas.", "Columnas inesperadas en CSV limpio.")
